# 04 - City-Level Hotspot Clustering

This notebook groups cities by AQI and pollutant behavior to identify pollution hotspot profiles.

In [ ]:
from pathlib import Path
import os

os.environ.setdefault('LOKY_MAX_CPU_COUNT', '1')

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid')

INPUT_PATH = Path('data/processed/combined_all_cities.csv')
REPORTS_DIR = Path('outputs/reports')
FIG_DIR = Path('outputs/figures')
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)


## Step 1 - Load Data

In [ ]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(f'Missing input file: {INPUT_PATH}')

df = pd.read_csv(INPUT_PATH)
required_cols = ['Timestamp', 'City', 'AQI', 'PM2.5', 'PM10']
missing = [col for col in required_cols if col not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
df = df.dropna(subset=['Timestamp', 'City']).copy()
for col in ['AQI', 'PM2.5', 'PM10']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print('Loaded shape:', df.shape)
print('Date range:', df['Timestamp'].min(), 'to', df['Timestamp'].max())
print('Cities:', sorted(df['City'].unique()))


## Step 2 - Feature Engineering (City Level)

In [ ]:
def month_to_season(month):
    if month in (12, 1, 2):
        return 'winter'
    if month in (6, 7, 8):
        return 'summer'
    if month in (9, 10, 11):
        return 'monsoon'
    return 'spring'

df['month'] = df['Timestamp'].dt.month
df['season'] = df['month'].apply(month_to_season)

base_features = df.groupby('City').agg(
    avg_AQI=('AQI', 'mean'),
    max_AQI=('AQI', 'max'),
    min_AQI=('AQI', 'min'),
    std_AQI=('AQI', 'std'),
    avg_PM2_5=('PM2.5', 'mean'),
    avg_PM10=('PM10', 'mean'),
).reset_index()

seasonal = (
    df[df['season'].isin(['winter', 'summer', 'monsoon'])]
    .groupby(['City', 'season'])['AQI']
    .mean()
    .unstack('season')
    .rename(columns={
        'winter': 'avg_AQI_winter',
        'summer': 'avg_AQI_summer',
        'monsoon': 'avg_AQI_monsoon',
    })
    .reset_index()
)

city_features = base_features.merge(seasonal, on='City', how='left')
city_features = city_features.fillna(city_features.median(numeric_only=True))
city_features


## Steps 3-5 - Normalize And Cluster

In [ ]:
feature_cols = [col for col in city_features.columns if col != 'City']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(city_features[feature_cols])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
city_features['Cluster'] = kmeans.fit_predict(X_scaled)
silhouette = silhouette_score(X_scaled, city_features['Cluster'])
print(f'Silhouette score for k=3: {silhouette:.3f}')

cluster_rank = (
    city_features.groupby('Cluster')['avg_AQI']
    .mean()
    .sort_values()
    .index
    .tolist()
)
cluster_name_map = {
    cluster_rank[0]: 'Relatively Cleaner Cities',
    cluster_rank[1]: 'Intermittent Hotspots',
    cluster_rank[2]: 'Severe Hotspots',
}
city_features['Cluster_Name'] = city_features['Cluster'].map(cluster_name_map)

city_features.sort_values(['Cluster', 'avg_AQI'], ascending=[True, False])


## Step 6 - Visualization

In [ ]:
plt.figure(figsize=(12, 7))
scatter = sns.scatterplot(
    data=city_features,
    x='avg_AQI',
    y='std_AQI',
    hue='Cluster',
    palette='Set2',
    s=150,
)
for _, row in city_features.iterrows():
    plt.text(row['avg_AQI'] + 1, row['std_AQI'] + 0.5, row['City'], fontsize=9)

plt.title('City Hotspot Clusters: Average AQI vs AQI Variability')
plt.xlabel('Average AQI')
plt.ylabel('AQI Standard Deviation')
plt.legend(title='Cluster')
plt.tight_layout()
plt.savefig(FIG_DIR / 'city_clusters_scatter.png', dpi=200)
plt.close()
print('Saved:', FIG_DIR / 'city_clusters_scatter.png')


## Step 7 - Interpretation

Cluster interpretation is based on average AQI, AQI variability, and seasonal AQI behavior:
- **Severe Hotspots**: consistently high AQI with extreme winter peaks; critical air quality risk zones.
- **Intermittent Hotspots**: moderate-to-high AQI with significant variability; episodic pollution spikes influenced by seasonal and regional factors.
- **Relatively Cleaner Cities**: lower average AQI and better dispersion conditions, while still showing occasional pollution variability.

In [ ]:
cluster_summary = city_features.groupby(['Cluster', 'Cluster_Name']).agg(
    city_count=('City', 'count'),
    cities=('City', lambda values: ', '.join(sorted(values))),
    avg_AQI=('avg_AQI', 'mean'),
    std_AQI=('std_AQI', 'mean'),
    avg_PM2_5=('avg_PM2_5', 'mean'),
    avg_PM10=('avg_PM10', 'mean'),
    avg_AQI_winter=('avg_AQI_winter', 'mean'),
    avg_AQI_summer=('avg_AQI_summer', 'mean'),
    avg_AQI_monsoon=('avg_AQI_monsoon', 'mean'),
).reset_index()
cluster_summary['Silhouette_k3'] = silhouette

display(cluster_summary)

for _, row in cluster_summary.iterrows():
    print(f"Cluster {int(row['Cluster'])} - {row['Cluster_Name']}: {row['cities']}")
    print(f"  Mean AQI={row['avg_AQI']:.2f}, Mean AQI std={row['std_AQI']:.2f}, Winter AQI={row['avg_AQI_winter']:.2f}")

summary_output = REPORTS_DIR / 'cluster_summary.csv'
cluster_summary.to_csv(summary_output, index=False)
print('Saved:', summary_output)


## Step 8 - Save Output

In [ ]:
output_path = REPORTS_DIR / 'city_clusters.csv'
city_features.to_csv(output_path, index=False)
print('Saved:', output_path)
print('Output shape:', city_features.shape)
